In [1]:
# !mkdir /food
# !cd food
# !wget http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
# !tar xzvf food-101.tar.gz

In [2]:
!nvidia-smi
!nvcc --version

Wed Apr  8 20:48:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.52.01              Driver Version: 591.74         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 Ti     On  |   00000000:06:00.0  On |                  N/A |
| 73%   51C    P5             24W /  220W |     677MiB /   8192MiB |     24%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!cd
!ls

10k.md	 CNN	    best_model_phase1.keras  food_cat.ipynb
10k.typ  CNN.ipynb  desktop.ini		     history_phase1.log
CNC.md	 CNN.md     food.ipynb		     tf-vla


In [4]:
!pip install tensorflow keras split-folders scikit-learn scikit-image seaborn opencv-python-headless #opencv-python


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
# !!pip uninstall -y opencv-python opencv-contrib-python
# !pip install opencv-python-headless


In [6]:
import tensorflow as tf
import subprocess
import os
subprocess.run(['ln', '-sf', '/usr/lib/x86_64-linux-gnu/libcudnn.so.8',
                '/usr/lib/x86_64-linux-gnu/libcudnn.so.9'], check=True)
subprocess.run(['ldconfig'], check=True)
# Check what's actually missing
result = subprocess.run(['ldconfig', '-p'], capture_output=True, text=True)
cudnn_libs = [l for l in result.stdout.split('\n') if 'cudnn' in l]
print('\n'.join(cudnn_libs))

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2026-04-08 20:48:02.383198: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775681282.397547    1161 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775681282.401946    1161 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-08 20:48:02.422053: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


	libcudnn_ops_train.so.8 (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_ops_train.so.8
	libcudnn_ops_infer.so.8 (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_ops_infer.so.8
	libcudnn_ops.so.9 (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_ops.so.9
	libcudnn_ops.so (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_ops.so
	libcudnn_heuristic.so.9 (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_heuristic.so.9
	libcudnn_heuristic.so (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_heuristic.so
	libcudnn_graph.so.9 (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_graph.so.9
	libcudnn_graph.so (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_graph.so
	libcudnn_engines_runtime_compiled.so.9 (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_engines_runtime_compiled.so.9
	libcudnn_engines_runtime_compiled.so (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_engines_runtime_compiled.so
	libcudnn_engines_precompiled.so.9 (libc6,x86-64) => /lib/x86_64-linux-gnu/libcudnn_engines_precompiled.so

In [7]:
import os
import shutil
import stat
import seaborn as sns
import collections
import h5py
import numpy as np
import tensorflow as tf
import matplotlib.image as img
import random
import cv2
import PIL
import matplotlib.pyplot as plt
import matplotlib.image as img
from os import listdir
from os.path import isfile, join
from collections import defaultdict
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
from sklearn.model_selection import train_test_split
import keras
from skimage.io import imread
#from keras.utils.np_utils import to_categorical
from keras.applications.inception_v3 import preprocess_input
from keras.models import load_model
from shutil import copy
from shutil import copytree, rmtree
import tensorflow as tf
# import tensorflow.keras.backen
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, EarlyStopping, ReduceLROnPlateau

print(tf.config.list_physical_devices('GPU'))
!python -c "import tensorflow as tf; print(tf.__version__)"

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
2026-04-08 20:48:06.730445: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775681286.743775    1230 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775681286.748060    1230 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-08 20:48:06.763324: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2.18.0


In [8]:
!ls /tf/data/food-101

README.txt  images  license_agreement.txt  meta  test  train


In [9]:
EPOCHS = 30
IMG_SIZE = (299, 299)
BATCH_SIZE = 16
N_CLASSES = 101
AUTOTUNE = tf.data.AUTOTUNE

ENV = None
# ENV = "cloud"
ENV = "local"

if ENV == "cloud":
    if not os.path.isdir("/content"):
        raise FileNotFoundError("/content does not exist")
    if not os.path.isdir("/content/food-101"):
        %cd /content
        !wget http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
        !tar xzvf food-101.tar.gz
    DATA_ROOT = "/content/food-101"

elif ENV == "local":
    if not os.path.isdir("/tf/data"):
        raise FileNotFoundError("/tf/data does not exist")
    DATA_ROOT = "/tf/data/food-101"

elif os.path.isdir("/content"):
    if not os.path.isdir("/content/food-101"):
        %cd /content
        !wget http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
        !tar xzvf food-101.tar.gz
    DATA_ROOT = "/content/food-101"

elif os.path.isdir("/tf/data"):
    DATA_ROOT = "/tf/data/food-101"

else:
    raise FileNotFoundError("Could not find /content or /tf/data")

TRAIN_DIR = f"{DATA_ROOT}/train"
TEST_DIR = f"{DATA_ROOT}/test"
META_DIR = f"{DATA_ROOT}/meta"
IMAGES_DIR = f"{DATA_ROOT}/images"



In [10]:
class_N = {}
N_class = {}
with open(f'{DATA_ROOT}/meta/classes.txt', 'r') as txt:
    classes = [i.strip() for i in txt.readlines()]
    class_N = dict(zip(classes, range(len(classes))))
    N_class = dict(zip(range(len(classes)), classes))
    class_N = {i: j for j, i in N_class.items()}
class_N_sorted = collections.OrderedDict(sorted(class_N.items()))
print(class_N)

# Method to generate directory-file map.
def gen_dir_file_map(path):
    dir_files = defaultdict(list)
    with open(path, 'r') as txt:
        files = [i.strip() for i in txt.readlines()]
        for f in files:
            dir_name, id = f.split('/')
            dir_files[dir_name].append(id + '.jpg')
    return dir_files

# Method to recursively copy a directory.
def copytree(source, target, symlinks = False, ignore = None):
  if not os.path.exists(target):
      os.makedirs(target)
      shutil.copystat(source, target)
  data = os.listdir(source)
  if ignore:
      exclude = ignore(source, data)
      data = [x for x in data if x not in exclude]
  for item in data:
      src = os.path.join(source, item)
      dest = os.path.join(target, item)
      if symlinks and os.path.islink(src):
          if os.path.lexists(dest):
              os.remove(dest)
          os.symlink(os.readlink(src), dest)
          try:
              st = os.lstat(src)
              mode = stat.S_IMODE(st.st_mode)
              os.lchmod(dest, mode)
          except:
              pass
      elif os.path.isdir(src):
          copytree(src, dest, symlinks, ignore)
      else:
          shutil.copy2(src, dest)

# Train files to ignore.
def ignore_train(d, filenames):
  subdir = d.split('/')[-1]
  train_dir_files = gen_dir_file_map(f'{DATA_ROOT}/meta/train.txt')
  to_ignore = train_dir_files[subdir]
  return to_ignore

# Test files to ignore.
def ignore_test(d, filenames):
  subdir = d.split('/')[-1]
  test_dir_files = gen_dir_file_map(f'{DATA_ROOT}/meta/test.txt')
  to_ignore = test_dir_files[subdir]
  return to_ignore

# Method to load and resize images.
def load_images(path_to_imgs):
  resize_count = 0

  invalid_count = 0
  all_imgs = []
  all_classes = []

  for i, subdir in enumerate(listdir(path_to_imgs)):
      imgs = listdir(join(path_to_imgs, subdir))
      classN = class_N[subdir]
      for img_name in imgs:
          img_arr = cv2.imread(join(path_to_imgs, subdir, img_name))
          img_arr_rs = img_arr
          img_arr_rs = cv2.resize(img_arr, (200,200),interpolation=cv2.INTER_AREA)
          resize_count += 1
          im_rgb = cv2.cvtColor(img_arr_rs, cv2.COLOR_BGR2RGB)
          all_imgs.append(im_rgb)
          all_classes.append(classN)

  return np.array(all_imgs), np.array(all_classes)

# Method to generate train-test files.
def gen_train_test_split(path_to_imgs = f'{DATA_ROOT}/images' , target_path = DATA_ROOT):
  copytree(path_to_imgs, target_path + '/train', ignore=ignore_test)
  copytree(path_to_imgs, target_path + '/test', ignore=ignore_train)

# Method to load train-test files.
def load_test_data(path_to_test_imgs):
  X_test, y_test = load_images(path_to_test_imgs)
  return X_test, y_test

def load_train_data(path_to_train_imgs):
  X_train, y_train = load_images(path_to_train_imgs)
  return X_train, y_train,



{'apple_pie': 0, 'baby_back_ribs': 1, 'baklava': 2, 'beef_carpaccio': 3, 'beef_tartare': 4, 'beet_salad': 5, 'beignets': 6, 'bibimbap': 7, 'bread_pudding': 8, 'breakfast_burrito': 9, 'bruschetta': 10, 'caesar_salad': 11, 'cannoli': 12, 'caprese_salad': 13, 'carrot_cake': 14, 'ceviche': 15, 'cheesecake': 16, 'cheese_plate': 17, 'chicken_curry': 18, 'chicken_quesadilla': 19, 'chicken_wings': 20, 'chocolate_cake': 21, 'chocolate_mousse': 22, 'churros': 23, 'clam_chowder': 24, 'club_sandwich': 25, 'crab_cakes': 26, 'creme_brulee': 27, 'croque_madame': 28, 'cup_cakes': 29, 'deviled_eggs': 30, 'donuts': 31, 'dumplings': 32, 'edamame': 33, 'eggs_benedict': 34, 'escargots': 35, 'falafel': 36, 'filet_mignon': 37, 'fish_and_chips': 38, 'foie_gras': 39, 'french_fries': 40, 'french_onion_soup': 41, 'french_toast': 42, 'fried_calamari': 43, 'fried_rice': 44, 'frozen_yogurt': 45, 'garlic_bread': 46, 'gnocchi': 47, 'greek_salad': 48, 'grilled_cheese_sandwich': 49, 'grilled_salmon': 50, 'guacamole': 5

In [11]:
# Generate train-test files.
if not os.path.isdir(f'{DATA_ROOT}/test') and not os.path.isdir(f'{DATA_ROOT}/train'):
    gen_train_test_split()
    len_train = len(os.listdir(f'{DATA_ROOT}/train'))
    len_test = len(os.listdir(f'{DATA_ROOT}/test')) # Fixed path from food-100 to food-101
    print('Train/Test dirs generated: len_train ='+len_train+'len_test='+len_test)
else:
    print('train and test folders already exists.')
    len_train = len(os.listdir(f'{DATA_ROOT}/train'))
    len_test = len(os.listdir(f'{DATA_ROOT}/test'))
    print(len_train,len_test)

train and test folders already exists.
101 101


In [12]:
@tf.function
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    # Pad 10% then random crop back to IMG_SIZE (replaces RandomZoom/Translate)
    pad_h = int(IMG_SIZE[0] * 0.1)
    pad_w = int(IMG_SIZE[1] * 0.1)
    image = tf.image.pad_to_bounding_box(
        image, pad_h, pad_w,
        IMG_SIZE[0] + 2 * pad_h,
        IMG_SIZE[1] + 2 * pad_w
    )
    # Fix: Include batch dimension in random_crop size
    image = tf.image.random_crop(image, size=[tf.shape(image)[0], IMG_SIZE[0], IMG_SIZE[1], 3])
    return image, label
    

def build_dataset(directory, augment_data=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    # preprocess_input: scales [0,255] → [-1, 1] as InceptionV3 expects
    ds = ds.map(lambda x, y: (preprocess_input(x), y),
                num_parallel_calls=AUTOTUNE)

    # Added caching to a file
    # ds = ds.cache(f'/content/cache_{"train" if augment_data else "test"}')
    return ds.prefetch(AUTOTUNE)

train_ds = build_dataset(TRAIN_DIR, augment_data=True)
test_ds  = build_dataset(TEST_DIR,  augment_data=False)

Found 75750 files belonging to 101 classes.


I0000 00:00:1775681293.147941    1161 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:06:00.0, compute capability: 8.6


Found 25250 files belonging to 101 classes.


In [13]:
# TRAINING
base_model = InceptionV3(weights='imagenet', include_top=False,
                         input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
# training=False: keeps InceptionV3's BatchNorm layers in inference mode
# while frozen — prevents corrupting pretrained running statistics
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(N_CLASSES,
                       kernel_regularizer=regularizers.l2(0.005),
                       activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary(show_trainable=True)

print("\n=== Phase 1: Head training ===")
history_p1 = model.fit(
    train_ds, validation_data=test_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint('best_model_phase1.keras',
                        save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(patience=5, restore_best_weights=True),
        CSVLogger('history_phase1.log'),
    ]
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 299, 299, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ inception_v3 (Functional)   │ (None, 8, 8, 2048)    │ 21,802,784 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ global_average_pooling2d    │ (None, 2048)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 256)           │    524,544 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout (Dropout)           │ (None, 256)           │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense_1 (Dense)             │ (None, 101)           │     25,957 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 22,353,285 (85.27 MB)

 Trainable params: 550,501 (2.10 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

...

4735/4735 ━━━━━━━━━━━━━━━━━━━━ 224s 47ms/step - accuracy: 0.5055 - loss: 2.0087 - val_accuracy: 0.6006 - val_loss: 1.5868
Epoch 29/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5069 - loss: 2.0138

2026-04-08 22:34:41.091550: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 33561088 bytes after encountering the first element of size 33561088 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


4735/4735 ━━━━━━━━━━━━━━━━━━━━ 217s 46ms/step - accuracy: 0.5069 - loss: 2.0138 - val_accuracy: 0.6040 - val_loss: 1.5754
Epoch 30/30
4735/4735 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5082 - loss: 2.0034

2026-04-08 22:38:23.597850: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 33561088 bytes after encountering the first element of size 33561088 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


4735/4735 ━━━━━━━━━━━━━━━━━━━━ 221s 46ms/step - accuracy: 0.5082 - loss: 2.0034 - val_accuracy: 0.5992 - val_loss: 1.6054


In [1]:
# ── Phase 2: Fine-tuning ────────────────────────────────────────────────────
# Resilient to kernel restarts: reload model + re-extract base_model if needed
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger

try:
    base_model  # already in scope from Cell 10
except NameError:
    print("base_model not in scope — reloading from best_model_phase1.keras")
    model = keras.models.load_model('best_model_phase1.keras')
    # InceptionV3 is always the first layer of the functional model
    base_model = next(l for l in model.layers if isinstance(l, tf.keras.Model))

# Unfreeze top 50 layers, keep the rest frozen
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

FINE_TUNE_EPOCHS = 30
steps_per_epoch  = len(train_ds)
total_steps      = FINE_TUNE_EPOCHS * steps_per_epoch

# Cosine decay: smooth LR anneal over all epochs
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=total_steps,
    alpha=1e-6
)

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')]
)

# MixUp augmentation
@tf.function
def mixup(ds_one, ds_two):
    images_one, labels_one = ds_one
    images_two, labels_two = ds_two
    lam = tf.random.uniform([tf.shape(images_one)[0], 1, 1, 1], 0.2, 0.8)
    images = lam * images_one + (1.0 - lam) * images_two
    labels = lam[:, 0, 0, :1] * tf.cast(labels_one, tf.float32) \
           + (1.0 - lam[:, 0, 0, :1]) * tf.cast(labels_two, tf.float32)
    return images, labels

train_ds_mixed = (
    tf.data.Dataset.zip((
        train_ds.shuffle(1000, reshuffle_each_iteration=True),
        train_ds.shuffle(1000, reshuffle_each_iteration=True),
    ))
    .map(mixup, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

callbacks_ft = [
    ModelCheckpoint('best_model_finetuned.keras', save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7),
    CSVLogger('history_finetune.log'),
]

print("\n=== Phase 2: Fine-tuning (top-50 layers unlocked) ===")
history_ft = model.fit(
    train_ds_mixed,
    validation_data=test_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks_ft
)
model.save('food101_inceptionv3_final.keras')


2026-04-08 22:46:40.118959: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775688400.230276    4059 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775688400.262037    4059 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-08 22:46:40.527499: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


base_model not in scope — reloading from best_model_phase1.keras


I0000 00:00:1775688404.101942    4059 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:06:00.0, compute capability: 8.6


NameError: name 'train_ds' is not defined

In [ ]:
# ── Validation on TEST_DIR ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best fine-tuned model (resilient to kernel restart)
try:
    model
except NameError:
    print("Loading best_model_finetuned.keras ...")
    model = keras.models.load_model('best_model_finetuned.keras')

# Rebuild test_ds if needed
try:
    test_ds
except NameError:
    test_ds = build_dataset(TEST_DIR, augment_data=False)

# ── Collect predictions ──────────────────────────────────────────────────────
print("Running inference on test set ...")
y_true, y_pred = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ── Metrics ──────────────────────────────────────────────────────────────────
top1_acc = np.mean(y_true == y_pred)
# Top-5: check if true label is in top-5 predicted indices
y_pred_top5 = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    top5 = np.argsort(preds, axis=1)[:, -5:]
    y_pred_top5.extend(top5.tolist())

true_in_top5 = [y_true[i] in y_pred_top5[i] for i in range(len(y_true))]
top5_acc = np.mean(true_in_top5)

print(f"\n{'='*40}")
print(f"  Top-1 Accuracy : {top1_acc*100:.2f}%")
print(f"  Top-5 Accuracy : {top5_acc*100:.2f}%")
print(f"{'='*40}\n")

# ── Per-class report (top 20 worst classes) ──────────────────────────────────
class_names = [N_class[i] for i in range(N_CLASSES)]
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

per_class_f1 = {cls: report[cls]['f1-score'] for cls in class_names}
worst20 = sorted(per_class_f1.items(), key=lambda x: x[1])[:20]

print("20 worst-performing classes (by F1):")
for cls, f1 in worst20:
    print(f"  {cls:<30} F1={f1:.3f}")

# ── Confusion matrix (top-20 worst classes only, for readability) ─────────────
worst_indices = [class_names.index(c) for c, _ in worst20]
mask_true = np.isin(y_true, worst_indices)
cm = confusion_matrix(y_true[mask_true], y_pred[mask_true], labels=worst_indices)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[class_names[i] for i in worst_indices],
            yticklabels=[class_names[i] for i in worst_indices])
plt.title('Confusion Matrix — 20 Worst Classes')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Clear CACHE
# import subprocess
# result = subprocess.run(
#     'rm -rf /content/*.tempstate* /content/*.lockfile /content/food-101.tar.gz && df -h /content',
#     shell=True, capture_output=True, text=True
# )
# print(result.stdout)
# print(result.stderr)

In [ ]:
# # CLEAN DEAD FILES
# import subprocess

# # Show the biggest directories eating your disk
# result = subprocess.run(
#     ['du', '-sh', '--threshold=100M',
#      '/content/cache_train',
#      '/content/cache_test',
#      '/content/food-101',
#      '/content/drive/.shortcut-targets-by-id',
#      '/root/.keras',
#      '/tmp'],
#     capture_output=True, text=True
# )
# print(result.stdout)
# print(result.stderr)

# # Also show overall breakdown
# print("\n--- Top space consumers in /content ---")
# subprocess.run(['du', '-sh', '/content/*'], shell=False)
# result2 = subprocess.run('du -sh /content/* 2>/dev/null | sort -rh | head -20',
#                          shell=True, capture_output=True, text=True)
# print(result2.stdout)